# Validation lab

Is a result an edge or a fluke, and what would it do in bad times? Every candidate passes through the same steps:

1. **Power**: before running anything, can the test answer its question?
2. **Strategy card**: is the mean return distinguishable from zero, from the best of many tries, from shifted timing?
3. **Robustness**: parameter plateau, costs, start dates, environments, best months removed.
4. **Walk-forward selection and PBO**: does *choosing* the parameters work out of sample?
5. **Scenarios and stress**: crises, doubled volatility, correlations going to one, gaps.
6. **The lab's record**: every card goes into the ledger; false-discovery control across all of them.

The example is H14's trend model on NDX. Every card is written to the lab ledger (`quantlab_data/research/ledger.jsonl`);
re-running a cell with the same inputs does not count it twice.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.data import align
from quantlab.signals import library as lib
from quantlab.engine.components import Targets, apply_signal
from quantlab.engine.backtest import run_backtest, Sizer, EngineConfig
from quantlab.validation import power, crossval, robustness, scenarios, fdr, ledger
from quantlab.validation.strategy_card import card_backtest
from quantlab.validation.regime_card import regime_card
from quantlab.regimes.models import FeatureBlendRegime, PercentileRegime, VolatilityRegime, HMMRegime

pd.set_option("display.width", 200)
bars = pd.read_csv("../data/research_cache/h14/NDX.csv", index_col=0, parse_dates=True)
bars.columns = [c.lower() for c in bars.columns]
panel = align({"NDX": bars})
returns = bars["close"].pct_change().to_frame("NDX")

def run(fast=20, slow=100, cost_multiplier=1.0, market=panel, market_returns=returns):
    signal = lib.h14_trend(fast=fast, slow=slow)
    return run_backtest(Targets(apply_signal(signal, market)), market_returns, costs_bps=1.5, sizer=Sizer(0.10, span=20, cap=2.0),
                        config=EngineConfig(band=0.02, max_position=2.0, cost_multiplier=cost_multiplier))

## 1. Power

How much history does the card's gate (Newey-West t ≥ 2) need for a plausible Sharpe? A test with a low pass
probability cannot show an idea is wrong: a failure is expected even if it is right. State these numbers in the
pre-registration.

In [ ]:
years = len(bars) / 252
print(f"{years:.0f} years of NDX")
display(pd.DataFrame({s: power.power_report(years, s) for s in (0.2, 0.3, 0.5, 0.8)}).T.drop(columns="warning", errors="ignore").round(2))
grid = np.linspace(5, 60, 56)
for s in (0.3, 0.5, 0.8):
    plt.plot(grid, [power.pass_probability(s, y) for y in grid], label=f"true Sharpe {s}")
plt.axhline(0.8, color="grey", lw=0.5); plt.xlabel("years of daily data"); plt.ylabel("probability the card passes"); plt.legend(); plt.show()
print(power.power_report(years, 0.3).get("warning", "adequately powered"))

## 2. The strategy card

`card_backtest` cards an engine result: the net P&L is tested, the positions drive the timing placebo, and the charts
add attribution. The baseline is buy-and-hold at the same volatility target.

In [ ]:
primary = run()
hold = run_backtest(Targets(pd.DataFrame(1.0, index=bars.index, columns=["NDX"])), returns, costs_bps=1.5, sizer=Sizer(0.10, span=20, cap=2.0),
                    config=EngineConfig(band=0.02, max_position=2.0))
variants = {f"fast={f}, slow={s}": run(f, s) for f, s in [(10, 50), (20, 100), (40, 200)]}
card = card_backtest(primary, name="H14 trend on NDX", baseline=hold, variants=variants, family="validation demo")
print(card)
display(card.table)
card.plot(); plt.show()

## 3. Robustness

A real effect sits on a plateau. Here the grid is the trend's fast and slow windows; the chosen point should have
neighbours almost as good (plateau ratio near 1).

In [ ]:
plateau = robustness.parameter_plateau(lambda fast, slow: run(fast, slow), {"fast": [10, 20, 40], "slow": [60, 100, 150, 200]}, chosen={"fast": 20, "slow": 100})
display(plateau["table"].pivot(index="fast", columns="slow", values="Sharpe").round(2))
print("chosen:", {k: round(v, 2) if isinstance(v, float) else v for k, v in plateau["chosen"].items()})

In [ ]:
costs = robustness.cost_sensitivity(lambda m: run(cost_multiplier=m))
print("Sharpe by cost multiplier:"); display(costs["table"].round(2)); print("breakeven multiplier:", round(costs["breakeven multiplier"], 1))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
robustness.start_dates(primary.pnl).plot.bar(ax=axes[0], title="Sharpe from each start year on")
dependence = robustness.episode_dependence(primary.pnl)
dependence["Sharpe without best months"].plot(marker="o", ax=axes[1], title="Sharpe without the best k months")
plt.tight_layout(); plt.show()
print(f"share of all P&L from the best 1% of days: {dependence['share of P&L from best 1% of days']:.0%}")
display(robustness.subperiods(primary.pnl).round(2))
display(robustness.environments(primary.pnl, robustness.market_environments(returns["NDX"])).round(2))

## 4. Walk-forward selection and PBO

What would *choosing* the best window each year have earned on the following year? Compare with the best grid point
in hindsight. PBO near 0.5 or above means picking the best in-sample is no better than picking at random.

In [ ]:
candidates = plateau["sweep"].dropna()
chosen = crossval.select_walk_forward(candidates, crossval.walk_forward_splits(len(candidates), test=252, min_train=5 * 252))
print({k: round(v, 3) for k, v in chosen.items() if isinstance(v, float)}, "pick changes:", chosen["pick changes"])
chosen["picks"]["pick"].value_counts()

## 5. Scenarios and stress

In [ ]:
display(scenarios.historical(primary.pnl, benchmark=returns["NDX"]).round(3))
shocks = scenarios.stress(primary.held, primary.returns)
display(shocks["table"].round(3))

## 6. The lab's record

Every card above, and the regime cards below (the six detectors from the regime lab), is in the ledger. With many
cards, some pass by luck; the false-discovery table shows which survive across all of them (Benjamini-Hochberg;
Storey's estimate of the share of true nulls needs at least 20 cards).

The p-value each card contributes tests the claim that matters: for a strategy, the larger of its t-test p and its
timing placebo p, so a trend book that only earns the index's drift is not a discovery. A discovery still needs a
passing verdict to be used.

In [ ]:
trend_pnl = primary.pnl
reversion_pnl = run_backtest(Targets(apply_signal(lib.zscore_reversion(40), panel)), returns, costs_bps=1.5, sizer=Sizer(0.10, span=20, cap=2.0),
                             config=EngineConfig(band=0.02, max_position=2.0)).pnl
detectors = {"H14 feature blend": (FeatureBlendRegime(), "TREND", "RANGE"), "efficiency ratio": (PercentileRegime("er"), "TREND", "RANGE"),
             "|slope t|": (PercentileRegime("abs_slope_t"), "TREND", "RANGE"), "ADX": (PercentileRegime("adx"), "TREND", "RANGE"),
             "volatility (calm = trend)": (VolatilityRegime(), "CALM", "STRESSED"), "HMM (calm = trend)": (HMMRegime(), "CALM", "VOLATILE")}
for name, (model, a, b) in detectors.items():
    regime_card(model(bars), trend_pnl, reversion_pnl, state_a=a, state_b=b, name=f"{name} on NDX", family="regime detectors")
table = fdr.lab_fdr(0.10)
print(f"estimated share of true nulls (Storey pi0): {table.attrs.get('pi0', float('nan')):.2f}")
table

In [ ]:
display(ledger.lab_summary())
fdr.dashboard(); plt.show()

### The frozen holdout

Data from 2026-01-01 on is cut from every card. A finished candidate gets **one** logged look:

```python
with ledger.holdout_look("H15 final", approved_by="owner"):
    final = card_backtest(run_on_full_data(), name="H15 final")
```

A second look at the same candidate is refused unless `again="<reason>"` is given, and that is logged too.